**Попытка попробовать другую модель # 404 (последняя)**

В этом ноутбуке у меня наконец-то получилось победить XGBRanker, над которым я мучилась более 2-х месяцев, потому что он по ряду причин отказывался работать, а рабочей документации по нему почему-то отвратительно мало, информацию собирала по крупицам.

В итоге у меня получилось как обучить модель, так и провести её кросс-валидацию с помощью custom метрики - MAP@10, для вычисления которой пришлось написать несколько разных "обёрток". Также вычислила еще ряд метрик, применительно к пользователям, продуктам, а также к взаимодействиям пользователь-продукт. Часть из них уже использовала ранее для модели предсказания reorder (см. в папке other models). Прилагаю самый рабочий вариант сюда.

Точность модели в чистом виде уже больше 0.25, но, т.к. в условиях сдачи указана предпочтительность гибридного подхода решения задачи, была предприянята весьма успешная попытка объединения модели с ALS. Простой взвешенный подход помогает увеличить точность предсказания до более чем 0.29 по метрике MAP@10 по private score на kaggle. Комбинация 50-50 уже чудесно работает, но простой перебор выявил, что 70% ALS к 30% XGB - самый оптимальный вариант (0.292 - private score).

In [1]:
#Для визуального отображения прогресса выполнения кода
from tqdm import tqdm
#Для взаимодействия с ОС (здесь нужна для работы с директориями файлов)
import os
#Для работы с датафреймами
import pandas as pd
#Для вычислений
import numpy as np
#Для нормализации данных (приведения к единому диапазону)
from sklearn.preprocessing import MinMaxScaler
#Для работы с моделью
import tensorflow as tf
#Для оптимизации работы с GPU при обучении модели
import tensorflow.keras.backend as K
#Для подачи метрики в модель
from sklearn.metrics import make_scorer
#И еще для работы с моделью (кросс-валидация, деление на фолды и случайный подбор гиперпараметров)
from sklearn.model_selection import cross_val_score, KFold, RandomizedSearchCV
#Для модели бустинга
import xgboost as xgb
#Для модели ALS библиотеки surprise
from surprise import Dataset, Reader, BaselineOnly
#Для оптимизации использования памяти
from functools import reduce
# import cupy as cp
#Для копирования данных
import copy
#Для учета времени выполнения операций
import time
#Для сохранения моделей:
import pickle
# Garbage Collector для периодической очистки памяти
import gc                         
gc.enable()

2025-04-29 13:36:54.349067: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-04-29 13:36:54.349110: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-04-29 13:36:54.349647: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-29 13:36:54.353194: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-04-29 13:36:54.867091: W tensorflow/compiler/tf2

In [2]:
#Загрузим данные из файлов. Для начала создадим два датасета - "products" и "transactions", соответственно:

data_dir = './data/recsys/'

PRODUCTS_CSV_PATH = os.path.join(data_dir, 'products.csv')
TRANSACTIONS_CSV_PATH = os.path.join(data_dir, 'transactions.csv')

products = pd.read_csv(PRODUCTS_CSV_PATH)
transactions = pd.read_csv(TRANSACTIONS_CSV_PATH)

## Описание данных

Согласно описанию к заданию, данные состоят из 2 файлов:

* products.csv - товары с их харакретистиками

* transactions.csv - транзакции покупателей

### Файл products.csv
product_id - уникальный идентификатор товара

product_name - название товара

aisle_id - уникальный идентификатор подкатегории

department_id - уникальный идентификатор категории

aisle - название подкатегории

department - название категории

### Файл transactions.csv
order_id - уникальный идентификатор транзакции

user_id - уникальный идентификатор покупателя

order_number - номер транзакции в истории покупок данного пользователя

order_dow - день недели транзакции

order_hour_of_day - час совершения транзакции

days_since_prior_order - количество дней с совершения предыдущей транзакции данным пользователем

product_id - уникальный идентификатор товара

add_to_cart_order - номер под которым данный товар был добавлен в корзину

reordered - был ли товар "перезаказан"

*Проверим, насколько это соответствует правде и, заодно, посмотрим подробную информацию по данных, хранящихся в них.*

In [3]:
#Теперь объединим датафреймы по product_id, исключив ранее обсуждаемые товары:
main = transactions.merge(products, how = 'left')
#Оценим результат:
main.head()

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,add_to_cart_order,reordered,product_name,aisle_id,department_id,aisle,department
0,2539329,1,1,2,8,NaN,196,1.0,0.0,Soda,77,7,soft drinks,beverages
1,2539329,1,1,2,8,NaN,14084,2.0,0.0,Organic Unsweetened Vanilla Almond Milk,91,16,soy lactosefree,dairy eggs
2,2539329,1,1,2,8,NaN,12427,3.0,0.0,Original Beef Jerky,23,19,popcorn jerky,snacks
3,2539329,1,1,2,8,NaN,26088,4.0,0.0,Aged White Cheddar Popcorn,23,19,popcorn jerky,snacks
4,2539329,1,1,2,8,NaN,26405,5.0,0.0,XL Pick-A-Size Paper Towel Rolls,54,17,paper goods,household


In [4]:
users, products, interactions = main.user_id.nunique(), main.product_id.nunique(), main.shape[0]
 
print('# users: ', users)
print('# products: ', products)
print('# interactions: ', interactions)

# users:  100000
# products:  49465
# interactions:  26408073


## Вычисление рейтинга товара

In [5]:
#Отрежем нужные нам столбцы из main:
main_sub = main[['order_id', 'user_id', 'order_number','product_id', 'add_to_cart_order','reordered',"order_dow","order_hour_of_day",'days_since_prior_order']]

In [6]:
del [products,transactions,main]
gc.collect()

43

Не по всем продуктам есть к-во дней с даты последней покупки. Замена значения на ноль в данном случае не совсем адекватна 
(ноль в данном случае будет означать, что продукт в среднем покупается ОЧЕНЬ часто). Поэтому заменим значения 
на среднее по пользователю.

In [7]:
main_sub['days_since_prior_order']=main_sub['days_since_prior_order'].fillna(main_sub.groupby('user_id')['days_since_prior_order'].transform('mean'))

Далее по порядку вычислим коэффициенты заказов, которые мы вычисляли для ALS, потом разберемся с остальным.


Создадим отдельный столбец с коэффициентом номера заказа для каждого пользователя. Напомню, что это нужно, чтобы в итоге получить более репрезентативные данные - на ранних стадиях пользования сервисом человек только знакомится с продуктовым набором, поэтому его покупки скорее хаотичны, чем имеют какую-то тенденцию. Кроме того, предпочтения со временем могут поменяться, что также стоит учесть. Чтобы нивелировать эту хаотичность присовим всем покупкам коэффициент номера покупки в виде:

${log_{10} x + 1} $, где x - номер заказа. Таким образом мы добъемся плавного увеличения веса конкретного товара в зависимости от увеличения номера покупки - более "свежий" заказ получит чуть больший вес.

In [8]:
main_sub['order_num_coef'] = main_sub['order_number'].apply(lambda x: round((np.log10(x) + 1),2))

In [9]:
#Перераспределим веса порядка заказа(перевернем порядок заказа, позиции с 11 и выше примут отрицательные значения).
main_sub['add_to_cart_coef'] = 11-main_sub['add_to_cart_order']
#Теперь скоректируем его с учетом номера заказа:
main_sub['add_to_cart_coef'] = main_sub['add_to_cart_coef'] * main_sub['order_num_coef']
main_sub.drop(columns = ['order_num_coef'], inplace = True) #Удалим лишнее

Также посчитаем композитный показатель с учетом перезаказов, на случай если нам вдруг захочется как-то соединить текущую модель с ALS. Для XGB модели мы использовать композитный рейтинг не будем, т.к. для перезаказов у нас есть отдельная модель.

In [10]:
#Посчитаем коэффициент перезаказов c учетом add_to_cart_coef:
main_sub['reordered'] = main_sub['reordered'] * main_sub['add_to_cart_coef']
# Посчитаем рейтинг, как add_to_cart_coef с учетом регулярности покупки продукта покупателем (reordered)
# Доли показателей определим как 0.8 и 0.2 в результирующем соответственно.
main_sub['rating']  = (main_sub.add_to_cart_coef * 0.8) + (main_sub.reordered * 0.2)

In [11]:
#Найдем сумму по всем позициям в связке продукт-пользователь:
main_sub['add_to_cart_coef'] = main_sub.groupby(['user_id','product_id'])['add_to_cart_coef'].transform('sum')
main_sub['rating'] = main_sub.groupby(['user_id','product_id'])['rating'].transform('sum')

Мы будем пытаться предсказывать именно по этому показателю, поэтому следует учитывать, что бездумное удаление значений ниже нуля приведет к потере части важных данных. Следовательно имеет смысл заменить их на ноль.

In [12]:
main_sub['add_to_cart_coef'] = main_sub['add_to_cart_coef'].clip(lower = 0)
main_sub['rating'] = main_sub['rating'].clip(lower = 0)

In [13]:
#Воспользуемся MinMaxScaler для слишком больших значений рейтинга, причем сделаем это в группировке по пользователю:
autoscaler = MinMaxScaler(feature_range = (1,10))
def sc(row):
    return autoscaler.fit_transform(row.values.reshape(-1,1))

main_sub['rating'] = main_sub.groupby('user_id')['rating'].apply(sc).explode().values.astype(float)
main_sub['add_to_cart_coef'] = main_sub.groupby('user_id')['add_to_cart_coef'].apply(sc).explode().values.astype(float)

/tmp/ipykernel_61415/3000763920.py:6: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  main_sub['rating'] = main_sub.groupby('user_id')['rating'].apply(sc).explode().values.astype(float)
/tmp/ipykernel_61415/3000763920.py:7: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  main_sub['add_to_cart_coef'] = main_sub.groupby('user_id')['add_to_cart_coef'].apply(sc).explode().values.astype(float)


In [14]:
#Вчислим ряд коэффициентов в группировке пользователь+продукт
main_sub['pu_orders'] = main_sub.groupby(['user_id','product_id'])['order_number'].transform('count')
main_sub['max_orders'] = main_sub.groupby(['user_id','product_id'])['order_number'].transform('max')
main_sub['min_orders'] = main_sub.groupby(['user_id','product_id'])['order_number'].transform('min')
main_sub['pu_order_ratio'] = main_sub['pu_orders']/(main_sub['max_orders'] - main_sub['min_orders'] + 1)
main_sub['pu_median_dspo'] = main_sub.groupby(['user_id','product_id'])['days_since_prior_order'].transform('median')
main_sub['pu_median_cart_pos'] = main_sub.groupby(['user_id','product_id'])['add_to_cart_order'].transform('median')    
# up_grouped = main_sub.groupby(['user_id','product_id'], as_index = False)
# main_sub['pu_orders'] = up_grouped['order_number'].nunique()['order_number']
# main_sub['max_orders'] = up_grouped['order_number'].max()['order_number']
# main_sub['min_orders'] = up_grouped['order_number'].min()['order_number']
# main_sub['pu_order_ratio'] = main_sub['pu_orders']/(main_sub['max_orders'] - main_sub['min_orders'] + 1)
# main_sub['pu_median_dspo'] = up_grouped['days_since_prior_order'].median()['days_since_prior_order']
# main_sub['pu_median_cart_pos'] = up_grouped['add_to_cart_order'].median()['add_to_cart_order']

#Округлим float, и удалим лишние колонки
main_sub['pu_median_dspo'] = main_sub['pu_median_dspo'].apply(lambda x: round(x,2))
main_sub['pu_median_cart_pos'] = main_sub['pu_median_cart_pos'].apply(lambda x: round(x,2))
main_sub['pu_order_ratio'] = main_sub['pu_order_ratio'].apply(lambda x: round(x,2))
main_sub.drop(columns = ['max_orders','min_orders'], inplace = True)
main_sub.head()

,order_id,user_id,order_number,product_id,add_to_cart_order,reordered,order_dow,order_hour_of_day,days_since_prior_order,add_to_cart_coef,rating,pu_orders,pu_order_ratio,pu_median_dspo,pu_median_cart_pos
0,2539329,1,1,196,1.0,0.0,2,8,20.259259,10.000000,10.000000,10,1.0,20.13,1.0
1,2539329,1,1,14084,2.0,0.0,2,8,20.259259,1.131082,1.105312,1,1.0,20.26,2.0
2,2539329,1,1,12427,3.0,0.0,2,8,20.259259,8.042105,8.057709,10,1.0,20.13,2.5
3,2539329,1,1,26088,4.0,0.0,2,8,20.259259,1.476663,1.476298,2,1.0,17.63,4.5
4,2539329,1,1,26405,5.0,0.0,2,8,20.259259,1.524330,1.536135,2,0.5,24.63,5.0


In [15]:
#Сохраним итог
main_sub.to_csv('./data/recsys/mainsub_order_xgb1.csv', sep='\t', index=False)

## Train и test наборы данных

In [2]:
#Загрузка из файла:
main_sub = pd.read_csv('./data/recsys/mainsub_order_xgb1.csv', sep='\t')

In [3]:
#Разделим данные на train и testset. Мне показалось логичным  в test отправить данные последнего заказа каждого пользователя.
testset = main_sub[main_sub.groupby('user_id')['order_number'].transform('max') == main_sub['order_number']]
trainset = main_sub[main_sub.groupby('user_id')['order_number'].transform('max') != main_sub['order_number']]
set_len = trainset.shape[0] + testset.shape[0]
print('\n К-во строк в тренировочном сете: ', trainset.shape[0], '\n','В тестовом сете ', testset.shape[0])
print('\n В процентом соотношении: ', round(100*(trainset.shape[0]/set_len),1), '% к ', 
      round(100*(testset.shape[0]/set_len),1), '%')


 К-во строк в тренировочном сете:  25328932 
 В тестовом сете  1079141

 В процентом соотношении:  95.9 % к  4.1 %


In [4]:
#Заменим значения в целевой колонке тестсета на перевернутые по порядку номера добавления товаров в покупки
testset['rating'] = 11-testset['add_to_cart_order']
testset['rating'] = testset['rating'].clip(lower = 0)#заменим на нули позиции больше 10-й

/tmp/ipykernel_62800/4110832256.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  testset['rating'] = 11-testset['add_to_cart_order']
/tmp/ipykernel_62800/4110832256.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  testset['rating'] = testset['rating'].clip(lower = 0)#заменим на нули позиции больше 10-й


In [5]:
#Очистка памяти:
del main_sub
gc.collect()

69

In [6]:
def get_feature_by_user(df):
    res = list()
    for i, v in tqdm(df.groupby('user_id')):
        res.append(
            (
                i,
                len(v['product_id']),
                v['days_since_prior_order'].median(),
                v['order_number'].max(),
                v['add_to_cart_order'].max(),
                v['order_hour_of_day'].median(),
                v['order_dow'].median()
            )
        )
    
    res = pd.DataFrame(
        res,
        columns=[
            'user_id', 'u_prods_count', 'u_median_dspo' ,'u_max_orders', 'u_max_ordlen','u_median_hod', 'u_median_dow'
        ])
    res['u_median_dspo'] = res['u_median_dspo'].apply(lambda x: round(x,2))
    
    return res

In [7]:
def get_feature_by_product(df):
    res = list()
    for i, v in tqdm(df.groupby('product_id')):
        res.append(
            (
                i,
                len(v['user_id']),
                v['add_to_cart_order'].median(),
                v['reordered'].sum(),
                v['order_dow'].median(),
                v['order_hour_of_day'].median(),
                v['days_since_prior_order'].median()
            )
        )
    
    res = pd.DataFrame(
        res,
        columns=[
            'product_id', 'p_user_cnt','p_median_cart_pos','p_reordered_times', 'p_median_dow', 'p_median_hod','p_median_dspo'
        ])
    res['p_median_dspo'] = res['p_median_dspo'].apply(lambda x: round(x,2))

    return res

In [8]:
def get_model_features(mainset, col_list):
    df = mainset.groupby(['user_id', 'product_id'], as_index = False).agg({**{feats:'median' for feats in col_list}})
    X_u = get_feature_by_user(mainset)
    gc.collect()
    merged = reduce(lambda left, right: pd.merge(left, right, on='user_id', how='inner'), [X_u,df])
    print('User features merged')
    
    del [X_u,df]
    gc.collect()
    
    X_p = get_feature_by_product(mainset)
    gc.collect()
    merged = reduce(lambda left, right: pd.merge(left, right, on='product_id', how='inner'), [X_p,merged])
    print('Product features merged')
    
    del [X_p,mainset]
    gc.collect()
    
    merged.fillna(0, inplace=True)
    
    features_cols = list(merged.drop(columns=['user_id', 'product_id', 'rating']).columns)

#     query_list = merged['user_id'].value_counts()


    merged = merged.set_index(['user_id', 'product_id'])

#     query_list= query_list.sort_index()

    merged.sort_index(inplace=True)

    df_x = merged[features_cols]
    df_x['qid'] = df_x.index.get_level_values('user_id')

    df_y = merged['rating'].apply(np.int64)
    
    del merged
    gc.collect()
    
    return df_x, df_y


col_list = ['rating','pu_orders','pu_median_dspo','pu_order_ratio','pu_median_cart_pos']
# col_list = ['add_to_cart_coef','rating','pu_orders','pu_median_dspo','pu_order_ratio','pu_median_cart_pos']
X_train, y_train= get_model_features(trainset, col_list)
X_test, y_test = get_model_features(testset, col_list)

100%|████████████████████████████████████████████████████████████████████████| 100000/100000 [00:09<00:00, 10758.65it/s]


User features merged


100%|███████████████████████████████████████████████████████████████████████████| 49413/49413 [00:08<00:00, 5707.53it/s]


Product features merged


100%|████████████████████████████████████████████████████████████████████████| 100000/100000 [00:08<00:00, 11312.08it/s]


User features merged


100%|██████████████████████████████████████████████████████████████████████████| 36108/36108 [00:03<00:00, 10835.87it/s]


Product features merged


In [9]:
#Немного сократим использование памяти для ускорения вычислений и снижения нагрузки на память:
def reduce_mem_usage(df, verbose=True):
    numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
    start_mem = df.memory_usage().sum() / 1024**2    
    for col in df.columns:
        col_type = df[col].dtypes
        if col_type in numerics:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)  
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)    
    end_mem = df.memory_usage().sum() / 1024**2
    if verbose: print('Mem. usage decreased to {:5.2f} Mb ({:.1f}% reduction)'.format(end_mem, 100 * (start_mem - end_mem) / start_mem))
    return df

In [10]:
#Применим фуекцию к нашим данным:
X_train = reduce_mem_usage(X_train)
X_test = reduce_mem_usage(X_test)

Mem. usage decreased to 413.75 Mb (67.1% reduction)
Mem. usage decreased to 49.34 Mb (67.6% reduction)


In [11]:
del [trainset,testset]
gc.collect()

0

## Оптимизация и обучение модели

In [12]:
# Функция для вычисления средней точности по n-позициям (компутационно аналогична ранее написанной):
def apn(real_labels, predicted_labels, n=10):
    """
    Построчное вычисление average precision at n
    """
    
    # Проверка на уникальность всех значений в наборе данных:
    if len(set(real_labels)) != len(real_labels):
        raise ValueError("Values in real_labels are not unique")

    if len(set(predicted_labels)) != len(predicted_labels):
        raise ValueError("Values in predicted_labels are not unique")
    
    #Обрезка до n-значений:
    if n != 0:
        predicted_labels = predicted_labels[:n]

    hits = 0
    running_sum = 0

    for i, pred in enumerate(predicted_labels):
        k = i+1 # our rank starts at 1
    
        if pred in real_labels:
            hits += 1
            running_sum += hits/k

    return running_sum/len(real_labels)

# Функция для вычисления MAP_at_n (также аналогична ранее написанной):
def MAP_at_n(fact, predictions, n=10):
    """
    Вычисление mean average precision at n для всех позиций
    """
    return np.mean([apn(f,p, n = n) for f,p in zip(fact, predictions)])

#Для вычисления показателей наши данные нужно обработать - для предсказания необходимы не сами суммы 'рейтинга',
# а отсортированные по порядку покупки списки продуктов (по факту - real_labels и по предсказанию - predicted_labels)
def generate_vals_for_calc(fact, predicted):
    """
    Обработка данных для расчета MAP@n
    """
    _pred_df = copy.deepcopy(fact)
    _pred_df = _pred_df.reset_index()
    _pred_df.rename(columns={"rating": "true_label"}, inplace = True)
    _pred_df['predicted_label'] = predicted
    _pred_df.sort_values(by=['user_id','true_label'], ascending=[True,False], inplace = True)
    real_labels = _pred_df.groupby('user_id')['product_id'].apply(list).values
    _pred_df.sort_values(by=['user_id','predicted_label'], ascending=[True,False], inplace = True)
    predicted_labels = _pred_df.groupby('user_id')['product_id'].apply(list).values
    return real_labels, predicted_labels

#Т.к. в кросс-валидации несколько фолдов данных, их нужно последовательно подать на обработку и вычисление MAP,
#поэтому напишем еще эту функцию-"обёртку":
def mean_avg_precision_at_n(fact, predicted, n=10):
    """
    Вычисление среднего показателя по всем фолдам
    """
    if n is None:
        n = len(fact[0])
    fact_all_sessions, predicted_all_sessions = generate_vals_for_calc(fact, predicted)
    return MAP_at_n(fact_all_sessions, predicted_all_sessions, n=n)

# И, наконец, обернём это всё в make_scorer из scikit-learn, чтобы можно было подать это как score по умолчанию:
mean_avg_precision_at_10_score = make_scorer(
    mean_avg_precision_at_n, greater_is_better=True, n=10)

In [13]:
#Очистка перед вычислениями:
gc.collect()
K.clear_session()

In [14]:
#Пробное вычисление cross_val_score на срезе данных с использованием кросс-валидации и нашего MAP@10
model = xgb.XGBRanker(objective='rank:ndcg', 
            lambdarank_pair_method='topk',
            lambdarank_num_pair_per_sample = 20,
#             sampling_method = 'gradient_based',
#             grow_policy = 'lossguide',
            tree_method ='hist',
            device = 'cuda',
#             eval_metric=['ndcg@10'],
            n_estimators=20, random_state=123, 
            learning_rate=0.1)
kfold = KFold(n_splits=3, shuffle=False)
# fit_params={
#             'qid' : X_train[:1000].qid
#            }



score = cross_val_score(model, X_train[:1000], y_train[:1000], cv=kfold,  
#                         params=fit_params,
                        scoring=mean_avg_precision_at_10_score, verbose=False, n_jobs =-1).mean()
print("Current score: ", score)
K.clear_session()
gc.collect()

Current score:  0.25495021927483014


/home/nette/miniconda3/lib/python3.11/site-packages/xgboost/core.py:158: UserWarning: [13:38:29] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1738880369036/work/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)
/home/nette/miniconda3/lib/python3.11/site-packages/xgboost/core.py:158: UserWarning: [13:38:29] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1738880369036/work/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cud

125

In [15]:
#К сожалению, у меня не получается реализовать подбор параметров сразу на всем сете - не хватает памяти.
#Поэтому, основываясь на принципе Паретто, ограничимся 20% сета для этой задачи:
train_dims = round(np.ceil(X_train.shape[0]*0.2))

#Train-test split:
# X=X_train1.iloc[:train_dims]
X=X_train.iloc[:train_dims]
y=y_train.iloc[:train_dims]

#Обозначим к-во фолдов для поиска параметров и обучения модели:
kfold = KFold(n_splits=3, shuffle=True, random_state=42)

In [16]:
# Определим список параметров для подборки:
param_space = {
    #Размер шага. Более низкие значения делают модель более устойчивой к переобучению.
    'learning_rate' : [0.10, 0.20, 0.30, 0.35],
    #Максимальная глубина дерева. Увеличение параметра усложняет модель, шанс переобучения также увеличивается.
    'max_depth' : [ 9, 10, 11, 12, 13, 14, 15],
    #Минимальная сумма веса экземпляра для разделения. Большие значения предотвращают переобучение.
    'min_child_weight' : [ 8, 9, 10, 11, 12],
    #Этот параметр определяет количество деревьев в модели. 
    #Увеличение параметра обычно повышает производительность, при этом увеличиваются время обучения и использование памяти.
    #Часто используется вместе с learning_rate (меньшая скорость обучения сочетается с большим количеством деревьев).
    'n_estimators' : [40, 50, 55, 60, 75, 90, 100]
}

# Инициализируем модель для поиска лучших параметров
xgb_ranker = xgb.XGBRanker(objective='rank:ndcg', 
            lambdarank_pair_method='topk',
            lambdarank_num_pair_per_sample = 20,
#             sampling_method = 'gradient_based',
#             grow_policy = 'lossguide',
            tree_method ='hist',
            device = 'cuda',
            random_state=42)


rs_model=RandomizedSearchCV(xgb_ranker,param_distributions=param_space,n_iter=5,scoring=mean_avg_precision_at_10_score,
                            n_jobs=-1,cv=kfold, random_state=42, verbose=3)



# Запустим подбор параметров
rs_model.fit(X, y)

# Напечатаем лучшие параметры и показатель для них
print("Best parameters found:", rs_model.best_params_)
print("Best score found:", rs_model.best_score_)

#Посмотрим на средние тестовые значения показателя для всех сплитов cv
print(rs_model.cv_results_['mean_test_score'])

Fitting 3 folds for each of 5 candidates, totalling 15 fits


/home/nette/miniconda3/lib/python3.11/site-packages/xgboost/core.py:158: UserWarning: [11:41:33] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1738880369036/work/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)
/home/nette/miniconda3/lib/python3.11/site-packages/xgboost/core.py:158: UserWarning: [11:41:38] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1738880369036/work/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cud

Best parameters found: {'n_estimators': 75, 'min_child_weight': 12, 'max_depth': 11, 'learning_rate': 0.1}
Best score found: 0.47954995532509087
[0.47954996 0.47954996 0.47954996 0.47954996 0.47954996]
[CV 2/3] END learning_rate=0.2, max_depth=9, min_child_weight=11, n_estimators=75;, score=0.479 total time= 1.3min
[CV 3/3] END learning_rate=0.2, max_depth=9, min_child_weight=11, n_estimators=75;, score=0.479 total time= 1.4min
[CV 1/3] END learning_rate=0.2, max_depth=9, min_child_weight=11, n_estimators=75;, score=0.480 total time= 1.4min
[CV 3/3] END learning_rate=0.1, max_depth=12, min_child_weight=8, n_estimators=50;, score=0.479 total time= 1.4min
[CV 1/3] END learning_rate=0.1, max_depth=12, min_child_weight=8, n_estimators=50;, score=0.480 total time= 1.4min
[CV 2/3] END learning_rate=0.1, max_depth=12, min_child_weight=8, n_estimators=50;, score=0.479 total time= 1.4min
[CV 1/3] END learning_rate=0.2, max_depth=14, min_child_weight=10, n_estimators=50;, score=0.480 total time=

In [17]:
 rs_model.best_params_

{'n_estimators': 75,
 'min_child_weight': 12,
 'max_depth': 11,
 'learning_rate': 0.1}

In [18]:
rs_model.best_score_

0.47954995532509087

In [16]:
#Очистка перед вычислениями:
gc.collect()
K.clear_session()

In [17]:
#Обучим на итоговом наборе параметров:
xgb_ranker = xgb.XGBRanker(objective ='rank:ndcg', 
            lambdarank_pair_method ='topk',
            lambdarank_num_pair_per_sample = 15,
            n_estimators = 75,
            min_child_weight = 12,
            max_depth = 11,
            learning_rate = 0.1,
            eval_metric=['ndcg@10'],
            tree_method ='hist',
            device = 'cuda',
            random_state=42)


xgb_ranker.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    verbose =10
)

[0]	validation_0-ndcg@10:0.77083
[10]	validation_0-ndcg@10:0.77267
[20]	validation_0-ndcg@10:0.77329
[30]	validation_0-ndcg@10:0.77277
[40]	validation_0-ndcg@10:0.77185
[50]	validation_0-ndcg@10:0.77124
[60]	validation_0-ndcg@10:0.77109
[70]	validation_0-ndcg@10:0.77124
[74]	validation_0-ndcg@10:0.77142


XGBRanker(base_score=None, booster=None, callbacks=None, colsample_bylevel=None,
          colsample_bynode=None, colsample_bytree=None, device='cuda',
          early_stopping_rounds=None, enable_categorical=False,
          eval_metric=['ndcg@10'], feature_types=None, gamma=None,
          grow_policy=None, importance_type=None, interaction_constraints=None,
          lambdarank_num_pair_per_sample=15, lambdarank_pair_method='topk',
          learning_rate=0.1, max_bin=None, max_cat_threshold=None,
          max_cat_to_onehot=None, max_delta_step=None, max_depth=11,
          max_leaves=None, min_child_weight=12, missing=nan,
          monotone_constraints=None, multi_strategy=None, n_estimators=75,
          n_jobs=None, ...)

In [18]:
#Посмотрим на mean_avg_precision_at_10_score на кросс-валидации:
kfold = KFold(n_splits=3, shuffle=True, random_state=42)
score = cross_val_score(xgb_ranker, X_train, y_train, cv=kfold,scoring=mean_avg_precision_at_10_score, 
                        verbose=False, n_jobs =-1).mean()
print("Current score: ", score)
K.clear_session()
gc.collect()

/home/nette/miniconda3/lib/python3.11/site-packages/xgboost/core.py:158: UserWarning: [13:41:04] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1738880369036/work/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)
/home/nette/miniconda3/lib/python3.11/site-packages/xgboost/core.py:158: UserWarning: [13:41:04] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1738880369036/work/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cud

Current score:  0.4772229572955819


568

In [19]:
#Посмотрим на предсказания модели на X_test, по аналогии с тем, что мы делали раньше:
#Найдем y_pred:
y_pred = xgb_ranker.predict(X_test)

/home/nette/miniconda3/lib/python3.11/site-packages/xgboost/core.py:158: UserWarning: [13:41:15] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1738880369036/work/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)


In [20]:
#Соберем в один сет данные план-факт:
pred_set = y_test.reset_index()
pred_set.rename(columns = {'rating':'real_rating'}, inplace = True)
pred_set['predicted_rating'] = y_pred
pred_set.head()

,user_id,product_id,real_rating,predicted_rating
0,1,196,10,4.370981
1,1,10258,5,2.739680
2,1,12427,2,3.196665
3,1,13032,3,-1.664545
4,1,25133,6,2.041414


In [21]:
#Теперь повторим всё то, что делали ранее для сбора предсказаний по ALS
#Обрезка до 10 самых крупных значений:
#Напишем, наконец, для этого функцию:
def return_n_greatest(dataset,desired_column, end_col_name, n=10, to_submit = False):
    endset = dataset.merge(dataset
        #Сгруппируем по пользователям, найдем 10 макс значений предсказанного рейтинга
        .groupby('user_id')[desired_column].nlargest(n)
        #Получим новый df с  MultiIndex, котрый сбросим через reset_index
        .reset_index('user_id'),
        # в новом df нет колонки "product_id" поэтому необходимо объединить изначальный сет с полученным
    how='right') # при этом все строки, которых нет в новом df удалятся
    
    if to_submit:
        endset['product_id'] = endset['product_id'].astype(str)
        endset.groupby('user_id')['product_id'].agg(' '.join).reset_index(name='product_id')

    else:
    #Соберем все id продуктов в один столбец - pred_order:
        endset = endset.groupby('user_id')['product_id'].unique().reset_index(name=end_col_name)
#     endset.columns=['user_id',end_col_name]
    return endset

In [22]:
def return_result_df(dataset, n=10):
    pred_df = return_n_greatest(dataset,desired_column = 'predicted_rating', end_col_name = 'pred_order')
    fact_df = return_n_greatest(dataset,desired_column = 'real_rating', end_col_name = 'fact_order')
    result_df = fact_df.merge(pred_df, how = 'left')
    #Применим функцию MAP_at_n построчно:
    result_df['fact_order'] = result_df['fact_order'].astype(str)
    result_df['pred_order'] = result_df['pred_order'].astype(str) 
    result_df['MAP'] = result_df.apply(lambda x: MAP_at_n(x.fact_order, x.pred_order), axis=1)
    return result_df

In [23]:
result = return_result_df(pred_set)

In [24]:
sum(result.MAP)/len(result.user_id.unique())

0.4881007348452811

In [25]:
del [result,pred_set,y_pred]
gc.collect()

0

In [26]:
#Сохраним модель:
with open('xgb_ranker.pkl','wb') as f:
    pickle.dump(xgb_ranker,f)

In [27]:
xgb_pred = xgb_ranker.predict(X_train)#полуаем предсказание по всему срезу пользователей и товаров
xgb_predset = y_train.reset_index(name='real_rating')#переформатируем в нормальный датасет с адекватными столбцами
# pred_set.rename(columns = {'add_to_cart_coef':'real_rating'}, inplace = True)
xgb_predset['predicted_rating'] = xgb_pred
xgb_predset = return_n_greatest(xgb_predset,desired_column = 'predicted_rating', end_col_name = None, to_submit = True)

In [28]:
xgb_predset.head(15)

,user_id,product_id,real_rating,predicted_rating
0,1,196,10,4.397383
1,1,12427,8,3.203403
2,1,10258,7,2.748731
3,1,25133,6,2.049164
4,1,46149,3,-0.245443
5,1,13032,2,-1.641930
6,1,49235,2,-2.393173
7,1,26088,1,-2.611268
8,1,26405,1,-2.815268
9,1,13176,1,-2.906235


In [29]:
prediction_xgb_solo = xgb_predset.groupby('user_id')['product_id'].agg(' '.join).reset_index(name='product_id')
#Cохраним
prediction_xgb_solo.to_csv('./data/recsys/prediction_xgb_solo.csv',sep=',', index=False)

In [30]:
del [prediction_xgb_solo]
gc.collect()

0

В таком виде модель показывает результат больше 0.259 по private score. Попробуем комбинации с другими моделями.

In [31]:
#Попробуем посмотреть на результат после объединения с предсказанием по столбцу reordered
#Чтобы не повторять обучение модели, загрузим уже готовое предсказние:
reorder_df =  pd.read_csv('./data/recsys/basic_reorder_by_prod.csv',sep='\t')

In [32]:
xgb_pred = xgb_ranker.predict(X_train)#полуаем предсказание по всему срезу пользователей и товаров
xgb_predset = y_train.reset_index(name='real_rating')#переформатируем в нормальный датасет с адекватными столбцами
xgb_predset['predicted_rating'] = xgb_pred
xgb_predset = xgb_predset.set_index(['user_id', 'product_id'])
reorder_df = reorder_df.set_index(['user_id', 'product_id'])
#Теперь попробуем объединить всё:
ldf = [xgb_predset, reorder_df]
xgb_predset = reduce(lambda x,y: x.join(y, how = "left"), ldf)

In [33]:
xgb_predset = xgb_predset.reset_index()
xgb_predset['corrected_rating'] = xgb_predset['predicted_rating'] * xgb_predset['reorder_pred']
xgb_predset.head(5)

,user_id,product_id,real_rating,predicted_rating,reorder_pred,corrected_rating
0,1,196,10,4.397383,1.0,4.397383
1,1,10258,7,2.748731,1.0,2.748731
2,1,10326,1,-4.429884,0.0,-0.000000
3,1,12427,8,3.203403,1.0,3.203403
4,1,13032,2,-1.641930,1.0,-1.641930


In [34]:
xgb_predset_reorder = return_n_greatest(xgb_predset,desired_column = 'corrected_rating', end_col_name = None, to_submit = True)

In [35]:
prediction_xgb_reorder = xgb_predset_reorder.groupby('user_id')['product_id'].agg(' '.join).reset_index(name='product_id')
#Cохраним
prediction_xgb_reorder.to_csv('./data/recsys/prediction_xgb_reorder.csv',sep=',', index=False)

In [36]:
del [xgb_predset_reorder, prediction_xgb_reorder]
gc.collect()

0

Предсказание самого факта покупки сделало всё хуже. Попробуем объединить это с ALS.

In [37]:
train_als = y_train.reset_index()
test_als = y_test.reset_index()
train_als['rating'] =train_als['rating'].astype(np.int32)
test_als['rating'] =test_als['rating'].astype(np.int32)

In [38]:
reader = Reader(rating_scale=(1, 10)) # Зададим разброс оценок

trainset = Dataset.load_from_df(train_als, reader)
trainset = trainset.build_full_trainset()
testset = Dataset.load_from_df(test_als, reader)
testset = [testset.df.loc[i].to_list() for i in range(len(testset.df))]

In [39]:
#Установим модель с лучшим результатом:
bsl_options =  {'method': 'als', 'n_epochs': 20, 'reg_u': 18, 'reg_i': 6}
algo = BaselineOnly(bsl_options = bsl_options)

In [40]:
%%time
#Сделаем предсказание:
predictions = algo.fit(trainset).test(testset)

Estimating biases using als...
CPU times: user 14 s, sys: 8.26 ms, total: 14 s
Wall time: 14 s


In [41]:
#Соберем в сет:
appended_data = []
for i in predictions:
    appended_data.append(i)
pred_df = pd.DataFrame(appended_data, columns = ['user_id','product_id','real_rating','predicted_rating','details'])
pred_df.drop(columns = ["details"], inplace = True)

In [42]:
pred_df.rename(columns = {'predicted_rating':'als_predicted'}, inplace = True)
pred_df = pred_df.drop(columns = ['real_rating'])
del [trainset,testset,appended_data,predictions]
gc.collect()
pred_df.head()

,user_id,product_id,als_predicted
0,1,196,3.658461
1,1,10258,2.543154
2,1,12427,2.729915
3,1,13032,2.135673
4,1,25133,2.409780


In [43]:
#Поменяем тип на int
pred_df["user_id"] = pred_df.user_id.astype(np.int64)
pred_df["product_id"] = pred_df.product_id.astype(np.int64)
pred_df = reduce_mem_usage(pred_df)
xgb_predset = reduce_mem_usage(xgb_predset)

Mem. usage decreased to 10.29 Mb (58.3% reduction)
Mem. usage decreased to 130.72 Mb (65.9% reduction)


In [44]:
xgb_predset = xgb_predset.set_index(['user_id', 'product_id'])
pred_df = pred_df.set_index(['user_id', 'product_id'])
#Теперь попробуем объединить всё:
ldf = [xgb_predset, pred_df]
xgb_predset = reduce(lambda x,y: x.join(y, how = "left"), ldf)
xgb_predset = xgb_predset.reset_index()
xgb_predset.head()

,user_id,product_id,real_rating,predicted_rating,reorder_pred,corrected_rating,als_predicted
0,1,196,10,4.398438,1.0,4.398438,3.658203
1,1,10258,7,2.748047,1.0,2.748047,2.542969
2,1,10326,1,-4.429688,0.0,-0.000000,NaN
3,1,12427,8,3.203125,1.0,3.203125,2.730469
4,1,13032,2,-1.641602,1.0,-1.641602,2.134766


In [45]:
#Заменим 'пустышки' на нули:
xgb_predset.fillna(0, inplace = True)
#Удалим ненужное:
xgb_predset.drop(columns = ['corrected_rating'], inplace = True)
xgb_predset.head()

,user_id,product_id,real_rating,predicted_rating,reorder_pred,als_predicted
0,1,196,10,4.398438,1.0,3.658203
1,1,10258,7,2.748047,1.0,2.542969
2,1,10326,1,-4.429688,0.0,0.000000
3,1,12427,8,3.203125,1.0,2.730469
4,1,13032,2,-1.641602,1.0,2.134766


In [46]:
del [pred_df]
gc.collect()

0

In [47]:
xgb_predset['predicted_rating'] = xgb_predset['predicted_rating'].apply(lambda x: round(x,2))
xgb_predset['als_predicted'] = xgb_predset['als_predicted'].apply(lambda x: round(x,2))
xgb_predset.head()

,user_id,product_id,real_rating,predicted_rating,reorder_pred,als_predicted
0,1,196,10,4.40,1.0,3.66
1,1,10258,7,2.75,1.0,2.54
2,1,10326,1,-4.43,0.0,0.00
3,1,12427,8,3.20,1.0,2.73
4,1,13032,2,-1.64,1.0,2.13


In [48]:
#Попробуем для начала простое 50/50 без учета предсказания reorder:
xgb_predset['combo_rating'] = (xgb_predset['predicted_rating'] + xgb_predset['als_predicted'])/2
xgb_predset.head(10)

,user_id,product_id,real_rating,predicted_rating,reorder_pred,als_predicted,combo_rating
0,1,196,10,4.40,1.0,3.66,4.030
1,1,10258,7,2.75,1.0,2.54,2.645
2,1,10326,1,-4.43,0.0,0.00,-2.215
3,1,12427,8,3.20,1.0,2.73,2.965
4,1,13032,2,-1.64,1.0,2.13,0.245
5,1,13176,1,-2.91,0.0,0.00,-1.455
6,1,14084,1,-3.88,0.0,0.00,-1.940
7,1,17122,1,-4.43,0.0,0.00,-2.215
8,1,25133,6,2.05,1.0,2.41,2.230
9,1,26088,1,-2.61,0.0,0.00,-1.305


In [49]:
xgb_predset_composite = return_n_greatest(xgb_predset,desired_column = 'combo_rating', end_col_name = None, to_submit = True)
xgb_predset_composite = xgb_predset_composite.groupby('user_id')['product_id'].agg(' '.join).reset_index(name='product_id')
#Cохраним
xgb_predset_composite.to_csv('./data/recsys/xgb_predset_c50-50.csv',sep=',', index=False)

Данный вариант пока что показывает лучшие результаты на Kaggle (точность чуть больше 0.28)

In [50]:
#40/60 без учета предсказания reorder:
xgb_predset['combo_rating2'] = xgb_predset['predicted_rating'] *0.4 + xgb_predset['als_predicted']*0.6
xgb_predset_composite = return_n_greatest(xgb_predset,desired_column = 'combo_rating2', end_col_name = None, to_submit = True)
xgb_predset_composite = xgb_predset_composite.groupby('user_id')['product_id'].agg(' '.join).reset_index(name='product_id')
#Cохраним
xgb_predset_composite.to_csv('./data/recsys/xgb_predset_c40-60.csv',sep=',', index=False)

Этот еще лучше (показатель ближе к 0.29)

In [51]:
#30/70 без учета предсказания reorder:
xgb_predset['combo_rating3'] = xgb_predset['predicted_rating'] *0.3 + xgb_predset['als_predicted']*0.7
xgb_predset_composite = return_n_greatest(xgb_predset,desired_column = 'combo_rating3', end_col_name = None, to_submit = True)
xgb_predset_composite = xgb_predset_composite.groupby('user_id')['product_id'].agg(' '.join).reset_index(name='product_id')
#Cохраним
xgb_predset_composite.to_csv('./data/recsys/xgb_predset_c30-70.csv',sep=',', index=False)

UPD: Этот самый  лучший (показатель больше 0.29 по private score).

В итоге наилучшим вариантом является наипростейший взвешенный (weighted) гибридный подход, основанный на объединении результатов предсказаний моделей XGBoost Ranker и ALS без использования предсказания факта покупки (reordered). Осталось собрать это всё в один класс для сдачи в соответствии с условиями задания.

In [53]:
#Сохраним ALS модель:
with open('als_model.pkl','wb') as f:
    pickle.dump(algo,f)